# BioListen VN — Anuraset Exploratory Data Analysis (EDA) [Direct Zip-Reading]

Notebook này thực hiện phân tích khám phá dữ liệu (EDA) cho bộ dữ liệu **AnuraSet** phục vụ việc mở rộng tập dữ liệu cho nhánh phân loại loài sinh vật (`species_head`).

### 💡 Giải pháp tối ưu hóa đĩa cứng (Direct In-Memory Zip-Reading):
Do bộ dữ liệu AnuraSet có kích thước rất lớn, việc giải nén toàn bộ tệp ZIP ra đĩa cứng của Google Colab dễ gây ra lỗi hết bộ nhớ đĩa (Disk Space Out of Memory). Để giải quyết triệt để vấn đề này, notebook sử dụng phương pháp **đọc trực tiếp từ tệp ZIP trong bộ nhớ (In-memory Zip-Reading)**:
- Mở tệp ZIP trực tiếp từ Google Drive sử dụng thư viện `zipfile` của Python.
- Đọc metadata CSV trực tiếp từ ZIP vào RAM bằng `pandas`.
- Khi cần trích xuất thuộc tính vật lý âm thanh hoặc trực quan hóa, hệ thống sẽ giải nén nhị phân tệp âm thanh WAV đó **trực tiếp vào bộ nhớ tạm (RAM) dưới dạng một luồng byte (`io.BytesIO`)**, nạp thông số qua `soundfile.info` mà **hoàn toàn không ghi bất cứ byte dữ liệu nào ra ổ đĩa của máy ảo**.

## 1. Kết nối Google Drive & Xác định Đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import io

zip_path = '/content/drive/MyDrive/Datasets/BioListenVN/raw_zips/anuraset.zip'

if os.path.exists(zip_path):
    print(f"Tìm thấy file ZIP dữ liệu tại: {zip_path}")
else:
    print(f"LỖI: Không tìm thấy file tại {zip_path}. Vui lòng kiểm tra lại Drive.")

## 2. Đọc Metadata CSV trực tiếp trong bộ nhớ RAM từ tệp ZIP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        if len(csv_files) > 0:
            # Đọc trực tiếp CSV từ luồng nhị phân của Zip
            with z.open(csv_files[0]) as f:
                df = pd.read_csv(f)
            print(f"Đã nạp thành công file metadata: {csv_files[0]}")
            print(f"Kích thước DataFrame: {df.shape}")
            print("\n5 dòng đầu tiên:")
            display(df.head())
            print("\nThông tin các cột:")
            df.info()
        else:
            print("Không tìm thấy file CSV metadata trong ZIP. Tự sinh DataFrame từ cấu trúc danh mục file WAV...")
            audio_files = [f for f in z.namelist() if f.endswith('.wav')]
            records = []
            for p in audio_files:
                records.append({
                    'file_name': os.path.basename(p),
                    'zip_member': p,
                    'label': os.path.basename(os.path.dirname(p))
                })
            df = pd.DataFrame(records)
            display(df.head())

## 3. Lập chỉ mục các file trong tệp ZIP và Phân phối nhãn loài

In [ ]:
# Lập chỉ mục đường dẫn file trong ZIP
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()
        audio_in_zip = [p for p in all_files if p.endswith('.wav')]
        file_to_zip_path = {os.path.basename(p): p for p in audio_in_zip}
        print(f"Lập chỉ mục thành công {len(audio_in_zip)} file WAV trong ZIP.")

# Tìm cột nhãn phù hợp
label_col = None
for col in ['label', 'class', 'species', 'species_id', 'Class Name']:
    if col in df.columns:
        label_col = col
        break

if not label_col and len(df.columns) > 0:
    label_col = df.columns[-1]

if label_col:
    class_counts = df[label_col].value_counts()
    print(f"Phân phối số lượng mẫu của các loài (theo cột '{label_col}'):")
    print(class_counts.head(20))
    
    plt.figure(figsize=(14, 8))
    sns.barplot(x=class_counts.values[:30], y=class_counts.index[:30], hue=class_counts.index[:30], palette="viridis", legend=False)
    plt.title("Phân phối số lượng mẫu của Top 30 Loài Ếch (AnuraSet)", fontsize=14, fontweight='bold')
    plt.xlabel("Số lượng mẫu")
    plt.ylabel("Nhãn loài")
    plt.show()

## 4. Thống kê đặc tính âm thanh nhanh (Audio Physical Profiling)

In [ ]:
import tqdm
import soundfile as sf
import librosa

# Lấy mẫu ngẫu nhiên 50 tệp để kiểm tra nhanh
sample_size = min(50, len(df))
sample_df = df.sample(sample_size, random_state=42)

durations = []
sample_rates = []
channels = []

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        for idx, row in tqdm.tqdm(sample_df.iterrows(), total=len(sample_df)):
            fname = row['file_name'] if 'file_name' in row else (row['filename'] if 'filename' in row else row.iloc[0])
            
            if fname in file_to_zip_path:
                zip_member = file_to_zip_path[fname]
                try:
                    with z.open(zip_member) as f:
                        # Đọc nhị phân vào BytesIO để nạp trong bộ nhớ
                        data_bytes = io.BytesIO(f.read())
                        info = sf.info(data_bytes)
                        sample_rates.append(info.samplerate)
                        durations.append(info.duration)
                        channels.append(info.channels)
                except Exception as e:
                    print(f"Lỗi đọc file {fname} từ ZIP: {e}")

if len(durations) > 0:
    print("\n--- KẾT QUẢ PHÂN TÍCH NHANH 50 FILE ANURASET (IN-MEMORY) ---")
    print(f"Tần số lấy mẫu (Sample Rates): {set(sample_rates)} Hz")
    print(f"Số kênh (Channels): {set(channels)}")
    print(f"Thời lượng ngắn nhất: {min(durations):.2f} giây")
    print(f"Thời lượng dài nhất: {max(durations):.2f} giây")
    print(f"Thời lượng trung bình: {np.mean(durations):.2f} giây")
    
    plt.figure(figsize=(8, 4))
    sns.histplot(durations, bins=15, kde=True, color='forestgreen')
    plt.title("Phân phối thời lượng tệp tin (AnuraSet)")
    plt.xlabel("Thời lượng (giây)")
    plt.ylabel("Tần suất")
    plt.show()
else:
    print("Không đọc được thông tin âm thanh nào từ các tệp thử nghiệm.")

## 5. Trực quan hóa âm học của một tiếng kêu mẫu

In [ ]:
import librosa.display
import IPython.display as ipd

def plot_frog_call_from_zip(zip_path, zip_member, label):
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(zip_member) as f:
            # Đọc trực tiếp vào RAM
            y, sr = sf.read(io.BytesIO(f.read()))
            # Nếu là nhiều kênh, lấy kênh đầu tiên
            if len(y.shape) > 1:
                y = y[:, 0]
                
    duration = len(y) / sr
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    
    fig, ax = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f"Tiếng kêu loài: {label} | File: {os.path.basename(zip_member)}", fontsize=14, fontweight='bold')
    
    librosa.display.waveshow(y, sr=sr, ax=ax[0], color='g')
    ax[0].set_title("Dạng sóng (Waveform)")
    ax[0].set_ylabel("Biên độ")
    
    img = librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis='time', y_axis='mel', ax=ax[1], cmap='viridis')
    ax[1].set_title("Log-Mel Spectrogram (dB Scale)")
    fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
    
    plt.tight_layout()
    plt.show()
    
    display(ipd.Audio(y, rate=sr))

# Chọn một dòng ngẫu nhiên để trực quan hóa
test_row = df.sample(1).iloc[0]
fname = test_row['file_name'] if 'file_name' in test_row else (test_row['filename'] if 'filename' in test_row else test_row.iloc[0])

if fname in file_to_zip_path:
    path_in_zip = file_to_zip_path[fname]
    plot_frog_call_from_zip(zip_path, path_in_zip, test_row[label_col])
else:
    print(f"Không tìm thấy đường dẫn tệp trong ZIP cho: {fname}")